# 🚀 ValCache — Integration Guide

**Async Redis Cache with Optional AES-128 Encryption**

This notebook walks you through every feature of ValCache:

| Section | Description |
|---------|-------------|
| **1. Installation** | Install ValCache and dependencies |
| **2. Key Generation** | Generate encryption keys and salts |
| **3. Normal Cache** | Plaintext caching with `ValCache` |
| **4. JSON Operations** | Store and retrieve dictionaries |
| **5. Encrypted Cache** | Transparent encryption with `EncryptedValCache` |
| **6. Hashed Keys** | Anonymize cache keys with PBKDF2 hashing |
| **7. Verify Encryption** | Prove that data is actually encrypted in Redis |
| **8. Health Check** | Monitor Redis connectivity |
| **9. Configuration** | `.env` file setup and environment variables |

---

## 1. Installation

Install ValCache with encryption support:

In [ ]:
# Normal mode only
# pip install valcache

# With encryption support (recommended)
!pip install valcache[encrypted]

## 2. Key Generation

Generate a secure Fernet encryption key and an optional salt for hashed keys.

In [ ]:
from cryptography.fernet import Fernet
import secrets

# Generate encryption key (save this in your .env file)
encryption_key = Fernet.generate_key().decode()
print(f"ENCRYPTION_KEY={encryption_key}")

# Generate a salt for hashed keys (save this too)
hash_salt = secrets.token_urlsafe(16)
print(f"HASH_SALT={hash_salt}")

---

## 3. Normal Cache — `ValCache`

Basic plaintext caching. Data is stored in Redis as-is.

**Prerequisites:** A running Redis server on `localhost:6379`.

In [ ]:
from valcache import ValCache

# Initialize cache (reads from .env or uses defaults)
cache = ValCache(host="localhost", port=6379, default_ttl=300)
await cache.connect()

print("✅ Connected to Redis")

In [ ]:
# SET — Store a value
await cache.set("user:1:name", "John Doe", ttl=60)
print("✅ Value stored")

# GET — Retrieve it
name = await cache.get("user:1:name")
print(f"Retrieved: {name}")

In [ ]:
# EXISTS — Check if key exists
exists = await cache.exists("user:1:name")
print(f"Key exists: {exists}")

# TTL — Check remaining time-to-live
ttl = await cache.get_ttl("user:1:name")
print(f"TTL remaining: {ttl}s")

In [ ]:
# DELETE — Remove a key
await cache.delete("user:1:name")

result = await cache.get("user:1:name")
print(f"After delete: {result}")  # None

In [ ]:
# Close connection
await cache.close()
print("✅ Connection closed")

---

## 4. JSON Operations

Store and retrieve entire dictionaries as JSON.

In [ ]:
cache = ValCache(default_ttl=300)
await cache.connect()

# Store a dictionary
profile = {
    "name": "Jane Smith",
    "email": "jane@example.com",
    "age": 28,
    "roles": ["admin", "editor"],
    "settings": {"theme": "dark", "notifications": True}
}

await cache.set_json("profile:42", profile)
print("✅ JSON stored")

# Retrieve it
result = await cache.get_json("profile:42")
print(f"Name: {result['name']}")
print(f"Roles: {result['roles']}")
print(f"Theme: {result['settings']['theme']}")

await cache.close()

---

## 5. Encrypted Cache — `EncryptedValCache`

All values are **automatically encrypted** (AES-128 Fernet) before being stored in Redis and **decrypted** on retrieval. The API is identical to `ValCache`.

Use the encryption key generated in Section 2, or set it in your `.env` file.

In [ ]:
from valcache import EncryptedValCache

# Pass encryption_key directly, or set ENCRYPTION_KEY in .env
encrypted_cache = EncryptedValCache(
    host="localhost",
    port=6379,
    default_ttl=300,
    encryption_key=encryption_key,  # from Section 2
)
await encrypted_cache.connect()

print("✅ Encrypted cache connected")

In [ ]:
# SET — Value is encrypted before storage
await encrypted_cache.set("secret:api_key", "sk-live-abc123xyz789")
print("✅ Encrypted value stored")

# GET — Value is decrypted on retrieval
api_key = await encrypted_cache.get("secret:api_key")
print(f"Decrypted: {api_key}")

In [ ]:
# JSON Encryption — Encrypt entire dictionaries
patient_record = {
    "name": "Alice Johnson",
    "ssn": "123-45-6789",
    "diagnosis": "Healthy",
    "blood_type": "O+",
    "medications": ["Vitamin D", "Iron"]
}

await encrypted_cache.set_json("patient:101", patient_record)
print("✅ Encrypted JSON stored")

# Retrieve and decrypt
record = await encrypted_cache.get_json("patient:101")
print(f"Patient: {record['name']}")
print(f"SSN: {record['ssn']}")
print(f"Meds: {record['medications']}")

---

## 6. Hashed Keys — Full Anonymization

With hashed keys, **even the Redis key itself is anonymized**. The key is deterministically hashed using PBKDF2-SHA256, and the value is encrypted with AES-128.

This is useful when the cache key contains sensitive data (e.g., patient emails, account numbers).

| What | Normal Encrypted | Hashed + Encrypted |
|------|-----------------|--------------------|
| Redis Key | `patient@hospital.com` (visible) | `a3F9x...Kz8=` (hashed) |
| Redis Value | `gAAAAABk...` (encrypted) | `gAAAAABk...` (encrypted) |

In [ ]:
SALT = hash_salt  # from Section 2

# Store with hashed key
await encrypted_cache.set_hashed(
    "patient@hospital.com",      # original key (will be hashed)
    "Diagnosis: All clear",       # value (will be encrypted)
    salt=SALT,
    ttl=600
)
print("✅ Stored with hashed key")

# Retrieve with same key + salt
result = await encrypted_cache.get_hashed("patient@hospital.com", salt=SALT)
print(f"Decrypted: {result}")

In [ ]:
# JSON with hashed keys
medical_data = {
    "blood_pressure": "120/80",
    "heart_rate": 72,
    "notes": "Patient is in good health"
}

await encrypted_cache.set_json_hashed("patient@hospital.com", medical_data, salt=SALT)
print("✅ JSON stored with hashed key")

result = await encrypted_cache.get_json_hashed("patient@hospital.com", salt=SALT)
print(f"Blood Pressure: {result['blood_pressure']}")
print(f"Heart Rate: {result['heart_rate']}")

In [ ]:
# Check existence and delete with hashed keys
exists = await encrypted_cache.exists_hashed("patient@hospital.com", salt=SALT)
print(f"Exists: {exists}")  # True

await encrypted_cache.delete_hashed("patient@hospital.com", salt=SALT)

exists = await encrypted_cache.exists_hashed("patient@hospital.com", salt=SALT)
print(f"After delete: {exists}")  # False

---

## 7. Verify Encryption

Let's prove that data is **actually encrypted** in Redis by reading raw values.

In [ ]:
# Store a known value
await encrypted_cache.set("verify:test", "PLAINTEXT_SECRET_123")

# Read the RAW value from Redis (bypass decryption)
client = await encrypted_cache._get_client()
raw_value = await client.get("verify:test")

print(f"Original:  PLAINTEXT_SECRET_123")
print(f"In Redis:  {raw_value[:60]}...")
print(f"")
print(f"Is plaintext visible? {'YES ❌' if 'PLAINTEXT' in raw_value else 'NO ✅'}")
print(f"Is Fernet token?      {'YES ✅' if raw_value.startswith('gAAAAA') else 'NO ❌'}")

# Now decrypt normally
decrypted = await encrypted_cache.get("verify:test")
print(f"")
print(f"Decrypted: {decrypted}")
print(f"Match?     {'YES ✅' if decrypted == 'PLAINTEXT_SECRET_123' else 'NO ❌'}")

In [ ]:
# Verify hashed keys — the key itself is anonymized
await encrypted_cache.set_hashed("user@example.com", "secret data", salt=SALT)

# Check all keys in Redis
all_keys = await encrypted_cache.keys("*")
print("Keys visible in Redis:")
for k in all_keys:
    print(f"  {k[:50]}{'...' if len(k) > 50 else ''}")

print(f"")
print(f"Is 'user@example.com' visible? {'YES ❌' if 'user@example.com' in all_keys else 'NO ✅'}")

---

## 8. Health Check

Verify that Redis is reachable.

In [ ]:
healthy = await encrypted_cache.health_check()
print(f"Redis status: {'🟢 Healthy' if healthy else '🔴 Unreachable'}")

In [ ]:
# Cleanup
await encrypted_cache.close()
print("✅ All connections closed")

---

## 9. Configuration via `.env`

The recommended way to configure ValCache is via a `.env` file in your project root. ValCache auto-loads it using `python-dotenv`.

### Example `.env` file

```env
# Redis Connection
REDIS_HOST=localhost
REDIS_PORT=6379
REDIS_DB=0
REDIS_PASSWORD=your_redis_password
REDIS_MAX_CONNECTIONS=20

# Encryption (for EncryptedValCache)
ENCRYPTION_KEY=your_fernet_key_here
```

### Priority Order

| Priority | Source | Example |
|----------|--------|--------|
| 1 (highest) | Constructor params | `ValCache(host="myhost")` |
| 2 | `.env` / environment | `REDIS_HOST=myhost` |
| 3 (lowest) | Built-in defaults | `localhost:6379` |

### Zero-config usage (with `.env`)

```python
from valcache import ValCache, EncryptedValCache

# Both read from .env automatically — no params needed
cache = ValCache()
encrypted_cache = EncryptedValCache()
```

---

## Quick Reference

### `ValCache` Methods

| Method | Description |
|--------|-------------|
| `connect()` | Initialize connection pool |
| `close()` | Close connection pool |
| `set(key, value, ttl)` | Store a value |
| `get(key)` | Retrieve a value |
| `delete(key)` | Delete a key |
| `exists(key)` | Check if key exists |
| `set_json(key, data, ttl)` | Store a dict as JSON |
| `get_json(key)` | Retrieve and parse JSON |
| `health_check()` | Ping Redis |
| `get_ttl(key)` | Get remaining TTL |
| `keys(pattern)` | List matching keys |
| `flush_db()` | Delete all keys |

### `EncryptedValCache` Additional Methods

| Method | Key | Value |
|--------|-----|-------|
| `set(key, value)` | Plaintext | 🔐 Encrypted |
| `get(key)` | Plaintext | 🔓 Decrypted |
| `set_hashed(key, value, salt)` | 🔒 Hashed | 🔐 Encrypted |
| `get_hashed(key, salt)` | 🔒 Hashed | 🔓 Decrypted |
| `delete_hashed(key, salt)` | 🔒 Hashed | — |
| `exists_hashed(key, salt)` | 🔒 Hashed | — |
| `set_json_hashed(key, data, salt)` | 🔒 Hashed | 🔐 Encrypted |
| `get_json_hashed(key, salt)` | 🔒 Hashed | 🔓 Decrypted |